# FinTextIntelligence: training run

This notebook fine-tunes XLM-R on the datasets listed in the PRD, checks the results on a held-out test set, and saves the trained model to my Google Drive.

Before running anything, switch the runtime to a GPU: **Runtime → Change runtime type → A100 GPU** if it's offered, otherwise T4. Training is about 86k rows: roughly 15–20 minutes on an A100 and about 70 on a T4.

The main run (steps 1–8) trains the **v3 setup**: run 2's settings plus native Arabic sentiment tweets (ArSarcasm-v2). Step 5 stops if the data isn't exactly what the v3 seeds trained and were tested on.

The code comes from Google Drive, so there's nothing to upload here. It looks for either:
- `FinTextIntelligence.zip` in My Drive (I replace it whenever the code changes), or
- the project folder, if it's synced from my Mac with Google Drive for desktop.

## 1. Check the GPU

Worth noting down what this prints. Any timing numbers only mean something alongside the hardware they came from.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

## 2. Get the project from Drive

This mounts Drive and finds the project. A synced folder wins if there is one, otherwise it uses the newest `FinTextIntelligence.zip` it can find. You can also set `PROJECT` or `PROJECT_ZIP` yourself.

If it comes from the zip, the code gets unpacked onto Colab's own disk, which is wiped when the session ends. So in that case:
- the trained model is saved to `My Drive/fintext-runs/`
- downloaded datasets are kept in `My Drive/fintext-raw/`, so the next session doesn't download them again

Either way, the resume checkpoint goes to `My Drive/fintext-checkpoints/` (it's about 3 GB and only useful here).

The XLM-R model itself is downloaded fresh each session onto Colab's disk. That takes about a minute, and it avoids a half-written copy on Drive breaking later runs.

In [ ]:
import glob, os, zipfile
from pathlib import Path
from google.colab import drive
from tqdm.auto import tqdm

drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive')

PROJECT = None       # e.g. '/content/drive/Othercomputers/My MacBook Pro/FinTextIntelligence'
PROJECT_ZIP = None   # e.g. '/content/drive/MyDrive/FinTextIntelligence.zip'

if PROJECT is None and PROJECT_ZIP is None:
    synced = glob.glob('/content/drive/Othercomputers/*/FinTextIntelligence')
    synced += glob.glob('/content/drive/Othercomputers/*/*/FinTextIntelligence')
    synced = [p for p in synced if Path(p, 'src', 'fintext').is_dir()]

    zips = glob.glob(str(DRIVE / '**' / 'FinTextIntelligence.zip'), recursive=True)
    zips.sort(key=os.path.getmtime, reverse=True)   # newest first

    if synced:
        PROJECT = synced[0]
    elif zips:
        PROJECT_ZIP = zips[0]
        if len(zips) > 1:
            print('found more than one FinTextIntelligence.zip, using the newest:', PROJECT_ZIP)
    else:
        raise SystemExit("Couldn't find the project on Drive. Put FinTextIntelligence.zip "
                         "in My Drive (or sync the project folder) and run this again.")

if PROJECT_ZIP is not None:
    # unpack to local disk, it's a lot faster than reading code off Drive
    with zipfile.ZipFile(PROJECT_ZIP) as archive:
        members = archive.infolist()
        bar = tqdm(total=sum(m.file_size for m in members), unit='B', unit_scale=True, desc='unzipping')
        for m in members:
            archive.extract(m, '/content')
            bar.update(m.file_size)
        bar.close()
    PROJECT = '/content/FinTextIntelligence'
    MODELS = DRIVE / 'fintext-runs'
    os.environ['FINTEXT_RAW_DIR'] = str(DRIVE / 'fintext-raw')
    print('using zip:', PROJECT_ZIP)
else:
    # synced folder: save the run straight into the project, if Colab is allowed to write there
    MODELS = Path(PROJECT) / 'artifacts' / 'models'
    test_file = MODELS / '.write-test'
    try:
        MODELS.mkdir(parents=True, exist_ok=True)
        test_file.write_text('ok')
        test_file.unlink()
    except OSError as e:
        MODELS = DRIVE / 'fintext-runs'
        print(f"can't write into the synced folder ({e}), saving runs to {MODELS} instead")

PROJECT = Path(PROJECT)
os.chdir(PROJECT)
CHECKPOINTS = DRIVE / 'fintext-checkpoints'
# HuggingFace downloads stay on Colab's own disk. A cache on Drive can end up with a
# half-written 1 GB model file, which then breaks every later run (SafetensorError).

print('project:', PROJECT)
print('models will be saved to:', MODELS)

## 3. Install packages

Colab already has torch, so this just adds the rest. I don't install the project itself; adding its `src/` folder to the path (next cell) is enough.

**HuggingFace token (optional, but downloads are faster with it):** click the 🔑 key icon in the left sidebar, add a secret called `fintext` with your HuggingFace token as the value, and switch on notebook access. The code picks it up from there. Don't paste the token into a cell.

In [ ]:
!pip install -q 'transformers>=4.40' 'datasets>=2.19' 'scikit-learn>=1.4' 'accelerate' 'tqdm'

## 4. Licence check and data sources

Nothing downloads until the licence check passes. This prints every dataset, where it actually comes from, and a note on how it's used. Good one to screenshot for the report.

Datasets that aren't on HuggingFace (OSACT4, HateXplain, Wikipedia Talk, ISOT, PHEME, FakeNewsNet, L-HSAB) get downloaded from their publishers the first time step 5 runs.

In [ ]:
import sys, json
sys.path.insert(0, str(PROJECT / 'src'))
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

from fintext import catalogue, corpora
from fintext.credentials import use_huggingface_token

source = use_huggingface_token()
print('HuggingFace token:', source or 'none found, downloads will be anonymous')

catalogue.check_training_sources(list(corpora.LOADERS))
print('licence check passed for:', ', '.join(corpora.LOADERS))
print()
for key, mirror in corpora.MIRRORS.items():
    print(f'{key:22} -> {mirror.path}  [{mirror.split}]')
    print(f'{"":22}    {mirror.note}')

## 5. Load the data

A few of the big toxicity and fake-news datasets are capped. Without the caps they'd swamp the sentiment data, and sentiment is what the trading gate actually uses. Arabic toxicity and all the sentiment data are loaded in full.

The data is split three ways: **train** to learn from, **dev** to pick the thresholds, and **test** for the final numbers. Nothing gets tuned on test.

Arabic sentiment training data comes from machine-translating the English sentiment sets. If `fintext-raw/translated_sentiment_ar/` isn't in your Drive yet, this step translates them first (about 6,000 sentences, a few minutes on the GPU) and saves the result there for next time.

Native Arabic sentiment also comes from **ArSarcasm-v2** (MIT licence, about 15,500 dialectal tweets), joined to train only, after the split. Any tweet whose normalised text matches a dev or test row is dropped first (6 of them). Dev and test are the same rows with or without it.

The last lines check two fingerprints. **Held-out** must be `23732d3d72c04f50`: the same dev and test rows as every run since run 2, so the results can be compared. **Train** must be `6acad74b3fe2f5f2`: the v3 seeds' train set. If either differs, the step stops before any GPU time is spent.

In [ ]:
rows, provenance = corpora.load_rows(
    tuple(corpora.LOADERS),
    eval_sources=tuple(corpora.EVAL_ONLY_LOADERS),    # ArSAS and L-HSAB: only ever evaluated on
    train_only=('translated_sentiment_ar',            # Arabic translations of the English sentiment data
                'arsarcasm_v2'),                      # native Arabic sentiment tweets (the v3 setup)
)
print(json.dumps(provenance, indent=2, ensure_ascii=False))

for split in ('train', 'dev', 'test'):
    arabic_sentiment = sum(1 for r in rows[split] if r['language'] == 'ar' and 'sentiment' in r['labels'])
    print(f'{split:5} {len(rows[split]):>6} rows   Arabic sentiment rows: {arabic_sentiment}')

# Arabic sentiment in train is the translated English data plus ArSarcasm-v2.
# In dev and test it's ArSAS, which is native Arabic and never trained on.
HELDOUT_FINGERPRINT = '23732d3d72c04f50'   # dev + test of every run since run 2
TRAIN_FINGERPRINT = '6acad74b3fe2f5f2'     # the v3 seeds' train set
totals = provenance['totals']
if totals['heldout_fingerprint'] != HELDOUT_FINGERPRINT:
    raise SystemExit(f"dev/test are {totals['heldout_fingerprint']}, not {HELDOUT_FINGERPRINT}: results wouldn't be comparable")
if totals['train_fingerprint'] != TRAIN_FINGERPRINT:
    raise SystemExit(f"train set is {totals['train_fingerprint']}, not the v3 setup's {TRAIN_FINGERPRINT}")
print('data matches the v3 setup: train', totals['train'], 'rows; dev/test unchanged')

## 6. Train

Three epochs, each with its own progress bar showing the batches done, the time left and the current loss. After training, two more bars appear while it scores the dev and test sets.

A checkpoint is saved after each epoch. If Colab disconnects, set `RUN_NAME` to the name printed below and run this cell again; it carries on from the last finished epoch.

In [ ]:
import logging, time
logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(message)s', force=True)
for noisy in ('httpx', 'huggingface_hub', 'urllib3'):   # one line per download request otherwise
    logging.getLogger(noisy).setLevel(logging.WARNING)

from fintext.training import TrainingConfig, train

RUN_NAME = f"shared-xlmr-{time.strftime('%Y%m%d-%H%M')}"   # dated so runs never clash
print('run name:', RUN_NAME)

config = TrainingConfig(
    epochs=3,
    batch_size=32,
    max_length=256,
    sources=tuple(corpora.LOADERS),
    run_name=RUN_NAME,
    output_dir=str(MODELS),
    checkpoint_dir=str(CHECKPOINTS),
    sentiment_weight=2.0,   # sentiment is only ~8% of the labelled rows, so give it more weight
)

started = time.time()
manifest = train(config, rows, provenance)
wall_minutes = (time.time() - started) / 60
print(json.dumps(manifest['metrics'], indent=2))

## 7. Results against the PRD targets

All of these numbers are from the test set. Thresholds were picked on dev.

`MISSING` means the metric wasn't measured in this run. Most of those are the service-level ones (latency, provenance and so on), which aren't measured during training. A missing metric still counts as a fail.

For toxicity, if no threshold gets both 0.90 recall and 0.90 precision, the model keeps the recall target and `toxicity_feasible_*` comes out `False`. The F1 shown is still the real F1 at that threshold.

In [ ]:
from fintext.evaluation import evaluate_acceptance

report = evaluate_acceptance(manifest['metrics'], model_version=config.run_name)

print(f"{'metric':34} {'measured':>9} {'target':>8}  status")
print('-' * 66)
for r in report.results:
    measured = '--' if r.measured is None else f'{r.measured:.3f}'
    target = '' if r.target is None else f'{r.comparison} {r.target}'
    print(f'{r.name:34} {measured:>9} {target:>8}  {r.status}')

print()
print('accepted:', report.accepted)
if report.failures:
    print('failing:', ', '.join(r.name for r in report.failures))

m = manifest['metrics']
print()
print('English sentiment F1:', m.get('sentiment_macro_f1_en'), '(target 0.80)')
print('Arabic sentiment F1: ', m.get('sentiment_macro_f1_ar'), '(reported, not gated; v3 seeds scored 0.730-0.738)')
print('EN/AR toxicity gap:  ', m.get('cross_language_f1_gap'), '(must be 0.07 or less)')
print('thresholds feasible: ', {k: m.get(f'toxicity_feasible_{k}') for k in ('en', 'ar')})

## 8. Getting the model onto the Mac

The model is already saved in the folder printed in step 2:
- **From the zip:** it's in `My Drive/fintext-runs/<run name>/`. In Google Drive, right-click that folder and choose **Download** (Drive zips it up). Then on the Mac, in the project folder:

  ```bash
  PYTHONPATH=src python3 -m fintext.cli import-model ~/Downloads/<the zip>.zip
  ```

- **From a synced folder:** it shows up in `artifacts/models/` on the Mac on its own.

This last cell just saves a note of the GPU, versions and run time next to the model. If you ever need everything as one file, set `MAKE_ZIP = True`.

After importing, the run is checked and registered on the Mac. The steps are in `docs/INTEGRATING_A_RUN.txt`: re-score, compare with the v3 seeds, leakage check, export, offsets, serving parity, benchmark. It registers as `shared-xlmr.v4`.

In [ ]:
import json, platform, zipfile
import transformers

run = Path(manifest['artifact_dir'])
print('run folder:', run)

notes = {
    'run_name': RUN_NAME,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
    'torch': torch.__version__,
    'transformers': transformers.__version__,
    'python': platform.python_version(),
    'wall_minutes': round(wall_minutes, 1),
}
(run / 'run_notes.json').write_text(json.dumps(notes, indent=2))
print(notes)

MAKE_ZIP = False
if MAKE_ZIP:
    # skip the checkpoint, it's only needed to resume training here
    files = [f for f in sorted(run.iterdir()) if f.name != 'checkpoint.pt']
    bundle = DRIVE / f'{RUN_NAME}.zip'
    bar = tqdm(total=sum(f.stat().st_size for f in files), unit='B', unit_scale=True, desc='zipping')
    with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_STORED) as z:   # weights don't compress anyway
        for f in files:
            info = zipfile.ZipInfo.from_file(f, f'{RUN_NAME}/{f.name}')
            with open(f, 'rb') as src, z.open(info, 'w', force_zip64=True) as dst:
                while block := src.read(8 * 1024 * 1024):
                    dst.write(block)
                    bar.update(len(block))
    bar.close()
    print(f'saved {bundle.name} to My Drive ({bundle.stat().st_size / 1e9:.2f} GB)')

## 9. Where toxicity goes wrong (optional)

The acceptance table gives one English and one Arabic toxicity number. This breaks them down by dataset, which is what decides the next change. It doesn't train anything.

To look at a model you trained earlier, without training again: run steps 1 to 5, skip 6 to 8, set `DIAGNOSE_RUN` below to its folder name in `My Drive/fintext-runs`, and run this. It checks that step 5 rebuilt exactly the training set that model saw. It takes about 3 minutes on a T4.

How to read the table:
- **AUC tr vs AUC held**: a big gap means the model memorised its training rows. Both low means it can't tell the classes apart on that dataset.
- **ceilF1**: the best F1 any threshold could get on that dataset. If it's under 0.80, no threshold choice can pass; the model itself has to get better.
- **F1@0.9R vs F1 dep**: F1 at 90% recall with that dataset's own threshold, against F1 at the one threshold the language actually uses. A big difference means datasets in the same language want different thresholds.
- **pos%**: how much of the dataset is toxic. Very different rates in one language pull its threshold in two directions.

The examples underneath are the model's most confident mistakes. If many of the "false positives" look clearly offensive to you, the labels are the problem, not the model.


In [ ]:
import torch
from transformers import AutoTokenizer
from fintext import diagnose
from fintext.training import TrainingConfig, build_model, pick_device

DIAGNOSE_RUN = globals().get('RUN_NAME') or 'shared-xlmr-20261118-1446'   # any folder in fintext-runs
run = MODELS / DIAGNOSE_RUN
recorded = json.loads((run / 'manifest.json').read_text())

# The breakdown is only meaningful on the split this model was trained and tuned on.
trained_on = recorded['corpus']['totals'].get('train_fingerprint')
rebuilt = provenance['totals']['train_fingerprint']
if trained_on != rebuilt:
    raise SystemExit(f'step 5 built a different train set ({rebuilt}) from the one '
                     f'{DIAGNOSE_RUN} was trained on ({trained_on}); not comparable')

device = pick_device()
model = build_model()
model.load_state_dict(torch.load(run / 'pytorch_model.bin', map_location='cpu'))
model.to(device).eval()
tokenizer = AutoTokenizer.from_pretrained(str(run))

table, scored = diagnose.toxicity_by_source(model, tokenizer, rows, TrainingConfig(), device)
print()
diagnose.print_table(table)

for language in ('ar', 'en'):
    errors = diagnose.worst_errors(scored, language=language, count=8)
    for kind, label in (('false_positives', 'labelled NOT toxic, model says toxic'),
                        ('false_negatives', 'labelled toxic, model says not')):
        print(f'\n{language}: {label}')
        for score, corpus, text in errors[kind]:
            print(f'  {score:.2f}  {corpus:14} {text[:110]}')


## 10. Seed runs (optional): how much does retraining alone move the numbers?

Run 2 beat run 1 on Arabic toxicity by about one AUROC point, but each was trained once. Two runs with identical settings also differ, just from the random start, so that point might be noise from training rather than the fixes. This step measures it.

It trains a setup three times, changing only the **training seed** (initial head weights, batch order, dropout). The data split stays exactly the same, so all the runs are tested on the same rows. `SETUP` picks which:

- `'v2'`: run 2's settings (already done: `shared-xlmr-v2-seed1..3`).
- `'run1'`: run 1's settings, from before the Arabic fixes. It trains on raw text with no extra Arabic weight. Comparing these with the v2 seeds shows whether the fixes changed anything.
- `'v3'`: run 2's settings plus native Arabic sentiment tweets (ArSarcasm-v2) in training. This step rebuilds the data for it (about a minute) and checks that dev and test are exactly run 2's.

- **Run steps 1 to 5 first, then skip straight to this one** (not 6 to 9).
- **Use an A100**: about 15 minutes per seed, so about 45 minutes in all. On a T4 it's about an hour per seed.
- **If Colab disconnects:** run steps 1 to 5 again, then this step. Seeds already finished are skipped, and only the one in progress starts over. Its checkpoint is on Colab's own disk, not Drive, which saves about 10 GB of Drive space.

**When it's done:** in Google Drive, download the **`fintext-seed-scores`** folder. It holds a few MB of per-row scores; the 1 GB model files aren't needed. Then note the zip's name.

Each seed's model is also saved in `fintext-runs/shared-xlmr-<SETUP>-seed<N>/`, about 1.1 GB each. If Drive gets full, those can go in the trash afterwards.

In [ ]:
import shutil, time
from fintext.training import TrainingConfig, train

SETUP = 'v3'     # 'v2' = run 2's settings; 'run1' = run 1's; 'v3' = v2 + native Arabic sentiment
SEEDS = [1, 2, 3]
assert SETUP in ('v2', 'run1', 'v3')
RUN_2_FINGERPRINT = '8ed9aa8269e9e3af'   # the train set run 1 and run 2 both saw
SCORES = DRIVE / 'fintext-seed-scores'
LOCAL_CHECKPOINTS = Path('/content/seed-checkpoints')

RUN_2_HELDOUT = '23732d3d72c04f50'      # run 2's dev + test rows
TRAIN_ONLY = ('translated_sentiment_ar', 'arsarcasm_v2') if SETUP == 'v3' else ('translated_sentiment_ar',)

setup_rows, setup_provenance = rows, provenance
if tuple(provenance['train_only']) != TRAIN_ONLY:
    print('rebuilding the data with', TRAIN_ONLY)
    setup_rows, setup_provenance = corpora.load_rows(
        tuple(corpora.LOADERS),
        eval_sources=tuple(corpora.EVAL_ONLY_LOADERS),
        train_only=TRAIN_ONLY,
    )

# Only meaningful on run 2's exact dev and test rows (and, except for v3, its train rows).
totals = setup_provenance['totals']
if totals.get('heldout_fingerprint') != RUN_2_HELDOUT:
    raise SystemExit(f"dev/test are {totals.get('heldout_fingerprint')}, not run 2's {RUN_2_HELDOUT}")
if SETUP != 'v3' and totals['train_fingerprint'] != RUN_2_FINGERPRINT:
    raise SystemExit(f"train set {totals['train_fingerprint']} is not run 2's {RUN_2_FINGERPRINT}")
print('train rows:', totals['train'], '| dev/test match run 2')

for seed in SEEDS:
    name = f'shared-xlmr-{SETUP}-seed{seed}'
    if (SCORES / name / 'scored_test.jsonl').exists():
        print(f'{name}: already done, skipping')
        continue
    print(f'\n=== {name} (training seed {seed}) ===')
    config = TrainingConfig(          # that run's settings, field for field, plus the seed
        epochs=3,
        batch_size=32,
        max_length=256,
        sources=tuple(corpora.LOADERS),
        run_name=name,
        output_dir=str(MODELS),
        checkpoint_dir=str(LOCAL_CHECKPOINTS),
        sentiment_weight=2.0,
        arabic_toxicity_weight=1.0 if SETUP == 'run1' else 1.5,
        train_on_model_text=(SETUP != 'run1'),
        training_seed=seed,
    )
    started = time.time()
    manifest = train(config, setup_rows, setup_provenance)
    run = Path(manifest['artifact_dir'])
    notes = {
        'run_name': name,
        'setup': SETUP,
        'training_seed': seed,
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
        'torch': torch.__version__,
        'wall_minutes': round((time.time() - started) / 60, 1),
    }
    (run / 'run_notes.json').write_text(json.dumps(notes, indent=2))

    out = SCORES / name
    out.mkdir(parents=True, exist_ok=True)
    for f in ('scored_test.jsonl', 'scored_dev.jsonl', 'manifest.json', 'run_notes.json'):
        shutil.copy(run / f, out / f)
    shutil.rmtree(LOCAL_CHECKPOINTS / name, ignore_errors=True)   # Colab's disk only

    m = manifest['metrics']
    print(f"{name}: AR AUROC {m.get('toxicity_auroc_ar'):.4f}  AR F1 {m.get('toxicity_f1_ar'):.4f}  "
          f"EN AUROC {m.get('toxicity_auroc_en'):.4f}  misleading {m.get('misleading_f1_en'):.4f}  "
          f"({notes['wall_minutes']} min)")

print('\ndone. Download this folder from Drive:', SCORES)

## 11. int8 on x86 (optional): which quantisation holds up on an Intel CPU?

The latency and throughput targets assume an **int8 model on an x86 server**. The first x86 run of this step showed two things:
- **The speed is there:** int8 is about 2× faster than fp32 on Colab's Xeon.
- **Default int8 isn't accurate enough:** on 1,000 held-out rows it flipped 90 sentiment labels, 41 toxicity flags and 48 misleading flags (`reports/int8_x86_colab.json`).

This run tries six other recipes against that default, on the same rows, to find one that keeps the speed without changing decisions:

| Recipe | What's quantised | Why |
|---|---|---|
| `int8` | everything (default) | control, the same as the first run |
| `int8_reduce_range` | everything, 7-bit weights | x86's known u8×s8 overflow fix |
| `uint8_weights` | everything, unsigned weights | the other x86 kernel path |
| `encoder_linear_only` | the 72 encoder linear layers | embeddings and output heads stay fp32 |
| `ffn_only` | the 24 feed-forward layers | attention stays fp32 |
| `ffn_only_reduce_range` | the same, plus 7-bit | both fixes together |
| `ffn_up_only` | the 12 first feed-forward layers | the second one's input has the largest outliers |

On the Mac (ARM), `ffn_up_only` flipped the fewest decisions (23 sentiment, 11 toxicity, 7 misleading) and `int8_reduce_range` the most (`reports/int8_arm_variants.json`). x86 int8 code differs, so this measures it there.

**Run steps 1 to 5, then skip straight to this one.** Any runtime works, because only the CPU is used. The A100 runtime has more cores and memory: the seven models take about 5 GB of memory between them. It takes about 30 minutes.

For each recipe it reports:
- the five-sentence probe check the registry uses (tolerance 0.05);
- the toxicity, misleading and sentiment decisions that flip at v4.1's thresholds and offsets, on 1,000 held-out rows, half of them Arabic;
- model-only p95 at 1 and 4 threads on 256-token inputs, against fp32.

A summary table is printed at the end.

**When it's done:** it saves `My Drive/fintext-x86/int8_x86_<time>.json`. Bring it back into the project from Drive.

Colab's CPU isn't the server the 50 ms target is written for, so read the timings as int8 against fp32 on x86. The flip counts carry over to any x86 machine.

In [ ]:
!pip install -q 'onnxruntime==1.30.0' 'onnx==1.22.0' 'onnxscript==0.7.2'   # the versions used on the Mac

import json, os, platform, random, subprocess, time
from pathlib import Path

import onnxruntime
import torch
from onnxruntime.quantization import QuantType, quantize_dynamic
from transformers import AutoTokenizer

from fintext.preprocess import model_text
from fintext.release_checks import quantisation_parity
from fintext.schemas import Language
from fintext.scoring import Onnx
from fintext.training import build_model, check_parity, export_onnx, measure_latency, row_key

RUN = 'shared-xlmr-20261119-1831'                        # shared-xlmr.v4, the current release
OFFSETS = {'ar': {'positive': 0.6, 'negative': 0.0, 'neutral': -0.2}}   # v4.1's Arabic offsets
SAMPLE = 1000                                             # held-out rows compared, half Arabic
THREADS = (1, 4)                                          # one core, and the four the service uses
WORK = Path('/content/int8-x86')                          # local disk: the ONNX files are ~1 GB

run_dir = MODELS / RUN
manifest = json.loads((run_dir / 'manifest.json').read_text())
thresholds = manifest['metrics']['thresholds']
if provenance['totals']['heldout_fingerprint'] != manifest['corpus']['totals']['heldout_fingerprint']:
    raise SystemExit('step 5 loaded different dev/test rows from the ones this run was scored on')

cpu = {'machine': platform.machine(), 'processor': platform.processor(), 'cores': os.cpu_count(),
       'onnxruntime': onnxruntime.__version__, 'torch': torch.__version__}
try:
    info = subprocess.run(['lscpu'], capture_output=True, text=True).stdout
    cpu['model'] = next((l.split(':', 1)[1].strip() for l in info.splitlines() if l.startswith('Model name')), '')
    flags = next((l for l in info.splitlines() if l.startswith('Flags')), '')
    cpu['int8_instructions'] = [f for f in ('avx2', 'avx512f', 'avx512_vnni', 'avx_vnni', 'amx_int8') if f in flags.split()]
except FileNotFoundError:
    cpu['model'] = platform.processor()
print('CPU:', cpu)

# 1. Export fp32 from the run's weights, on CPU, and quantise two ways.
WORK.mkdir(parents=True, exist_ok=True)
model = build_model(manifest['config']['encoder'])
model.load_state_dict(torch.load(run_dir / 'pytorch_model.bin', map_location='cpu', weights_only=False))
model.eval()
tokenizer = AutoTokenizer.from_pretrained(str(run_dir))
fp32_path = export_onnx(model, tokenizer, WORK / 'v4.onnx')
del model
# Which nodes are which: 12 layers x (query, key, value, attention output, two
# feed-forward) linear MatMuls, two attention-score MatMuls (never quantised),
# four head Gemms, and the embedding Gathers.
import onnx
graph = onnx.load(str(fp32_path), load_external_data=False).graph
FFN = [n.name for n in graph.node if n.op_type == 'MatMul'
       and ('/intermediate/dense/' in n.name or ('/output/dense/' in n.name and '/attention/' not in n.name))]
assert len(FFN) == 24, f'expected 24 feed-forward MatMuls, found {len(FFN)}'
FFN_UP = [n for n in FFN if '/intermediate/dense/' in n]   # 768 -> 3072; its input has no GELU outliers
del graph

RECIPES = {
    'int8':                  {},                                            # control: the default, as before
    'int8_reduce_range':     {'reduce_range': True},                        # 7-bit weights: the x86 u8 x s8 saturation fix
    'uint8_weights':         {'weight_type': QuantType.QUInt8},             # u8 x u8 kernels instead
    'encoder_linear_only':   {'op_types_to_quantize': ['MatMul']},          # embeddings and heads stay fp32
    'ffn_only':              {'nodes_to_quantize': FFN},                    # only the 24 feed-forward layers
    'ffn_only_reduce_range': {'nodes_to_quantize': FFN, 'reduce_range': True},
    'ffn_up_only':           {'nodes_to_quantize': FFN_UP},                 # only the first feed-forward layer
}
variants = {}
for name, options in RECIPES.items():
    variants[name] = WORK / f'v4.{name}.onnx'
    quantize_dynamic(str(fp32_path), str(variants[name]), **({'weight_type': QuantType.QInt8} | options))
    print(f'{name:22} {variants[name].stat().st_size / 1e6:6.0f} MB')
backends = {'fp32': Onnx(fp32_path, tokenizer, version='v4-fp32', threads=4, sentiment_offsets=OFFSETS)}
backends |= {k: Onnx(p, tokenizer, version=f'v4-{k}', threads=4, sentiment_offsets=OFFSETS) for k, p in variants.items()}

def caller(backend):
    return lambda text, language: backend.predict(model_text(text), Language(language))

# 2. Is this fp32 export the model the run's scores came from? (Colab's transformers may differ.)
scored = {}
for line in (run_dir / 'scored_test.jsonl').read_text().splitlines()[1:]:
    r = json.loads(line); scored[r['key']] = r
rng = random.Random(20261120)
test = rows['test']
arabic = [r for r in test if r['language'].startswith('ar')]
english = [r for r in test if r['language'] == 'en']
sample = rng.sample(arabic, SAMPLE // 2) + rng.sample(english, SAMPLE - SAMPLE // 2)
export_check = 0.0
for r in sample[:200]:
    s = backends['fp32'].predict(model_text(r['text']), Language(r['language'].split('-')[0]))
    k = scored[row_key(r)]
    for name, value in (('toxicity_score', s.toxicity), ('misleading_score', s.misleading)):
        if name in k:
            export_check = max(export_check, abs(k[name] - value))
print(f'fp32 export vs the run\'s own scores, 200 rows: max delta {export_check:.2e}')

# 3. int8 against fp32: the five-probe gate, and decisions on real held-out rows.
results = {'run': RUN, 'cpu': cpu, 'recipes': {k: {o: (str(v) if not isinstance(v, list) else f'{len(v)} nodes') for o, v in r.items()} for k, r in RECIPES.items()}, 'fp32_export_vs_run_scores': export_check, 'variants': {}}
for name in variants:
    probe = check_parity(backends['fp32'], backends[name])
    rows_check = quantisation_parity(caller(backends['fp32']), caller(backends[name]), sample, thresholds, OFFSETS)
    results['variants'][name] = {'probe_parity': probe, 'heldout_parity': rows_check}
    print(f"\n{name}: probe max delta {probe['max_delta']} -> {'PASS' if probe['ok'] else 'FAIL'} (tolerance {probe['tolerance']})")
    print(f"  on {rows_check['rows']} held-out rows: max {rows_check['max_delta']}")
    print(f"  p99 {rows_check['p99_delta']}")
    print(f"  decisions flipped: {rows_check['decision_flips']}")

# 4. Latency, model only, 256-token inputs, fp32 against int8 on this CPU.
results['latency'] = {}
for name, path in {'fp32': fp32_path, **variants}.items():
    for threads in THREADS:
        timed = measure_latency(Onnx(path, tokenizer, version=name, threads=threads), iterations=200)
        results['latency'][f'{name}@{threads}'] = timed
        print(f"{name:17} {threads} thread(s): p50 {timed['p50_ms']:.1f} ms, p95 {timed['p95_ms']:.1f} ms")

out_dir = DRIVE / 'fintext-x86'
out_dir.mkdir(parents=True, exist_ok=True)
out = out_dir / f"int8_x86_{time.strftime('%Y%m%d-%H%M')}.json"
out.write_text(json.dumps(results, indent=2))
print('\nsaved', out)

print('\nsummary: decisions flipped on', SAMPLE, 'held-out rows (fewer is better), and int8 / fp32 time at 4 threads')
fp32_ms = results['latency']['fp32@4']['p95_ms']
for name in variants:
    flips = results['variants'][name]['heldout_parity']['decision_flips']
    speed = fp32_ms / results['latency'][f'{name}@4']['p95_ms']
    print(f"  {name:22} toxicity {flips['toxicity']:3}  misleading {flips['misleading']:3}  sentiment {flips['sentiment']:3}   {speed:.2f}x faster")


## A note on latency

The PRD's 50 ms target is for a CPU, so the GPU's speed doesn't tell us much. Step 11 compares fp32 with int8 on Colab's x86 CPU. The real number needs `fintext bench --gate` on the target server, after exporting the model to ONNX.